# Fix a mosaic-shift-induced FOV gap

A worked example of diagnosing and patching a real acquisition-planning bug,
kept as a template for the same class of problem on other samples --
`notebooks/tests/` holds notebooks like this one, built to investigate/fix a
specific real issue rather than run as a standing part of the numbered
`prepare_imaging` pipeline.

**The bug this fixes**: the usual workflow is (1) scan a low-mag (10x) Steve
mosaic of the whole coverslip, (2) image a handful of FOVs at the real
imaging objective ("60x" here) over part of that scan to measure the fixed
stage-calibration offset between the two objectives, (3) shift the 10x
mosaic's stage positions by that offset so it lines up with real (60x)
coordinates, THEN (4) derive the tissue boundary and FOV grid from the
shifted mosaic. If step 3 is skipped (or the shift is computed but applied
too late, after the boundary was already saved), the boundary -- and every
FOV grid built from it -- ends up offset from where the tissue actually is.

**Sections**:
1. Load the raw Steve mosaic tiles (cached locally -- a slow, many-small-file
   read over a network drive) and composite BOTH objectives together
   (unshifted) into one image -- real tissue content, not just tile-center
   coordinates, so a real misalignment is visible right away.
2. Same composite, but with the known `(SHIFT_DX_UM, SHIFT_DY_UM)` correction
   applied to the low-mag tiles first -- the high-mag patch should now look
   like a seamless part of the surrounding tissue.
3. Assemble the shifted mosaic (low-mag only, real un-normalized values) and
   overlay the CURRENT (already-imaging) positions file's own FOV perimeters
   on top of it, to see the real-world misalignment directly.
4. Re-run tissue segmentation on the shifted mosaic (same parameters as this
   sample's own local `02_create_boundary_from_mosaic.ipynb`), then find FOV
   positions in that new boundary using the SAME dense grid the current
   positions file's own boundary was filtered from -- not an independently
   re-centered grid -- so any FOV the two boundaries share ends up at the
   EXACT same coordinate, not just an approximately-overlapping one.
5. Overlay the OLD vs. NEW tissue boundary outlines.
6. Classify every NEW FOV as already-covered (an exact coordinate match in
   the OLD grid) or MISSING, and count/report the missing ones.
7. Append the missing FOVs -- re-ordered into their own short-travel loop --
   to the end of the current positions array and save a new
   `positions_{tag}_added.txt`, ready to be imaged in a follow-up loop.

Does not touch the original positions file -- only ever writes a new,
distinctly-named one, and every plot is saved to `SAMPLE_DIR/figures/` for
visual review before trusting the result.

## 1 — Setup

In [ ]:
import os
import sys
import pickle
import dataclasses
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import box
from shapely.affinity import translate

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/tests/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io              import save_positions_array
from MERci.acquisition.configs    import get_fov_geometry
from MERci.acquisition.mosaic     import (
    load_steve_mosaic, assemble_mosaic_canvas, segment_mosaic_tissue, plot_mosaic_segmentation,
)
from MERci.acquisition.positions  import (
    load_boundary_polygon, load_hole_polygons, create_grid_positions,
    generate_scanning_path, filter_scanning_path, get_path_stats,
)

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
POSITIONS_DIR = SAMPLE_DIR / "positions"
FIGURES_DIR   = SAMPLE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_NAME = "fix_mosaic_shift_missing_fovs"
CACHE_DIR     = SAMPLE_DIR / "analysis" / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Mosaic source ──────────────────────────────────────────────────────────
MOSAIC_DIR  = SAMPLE_DIR / "data" / "mosaic10x"
MOSAIC_NAME = None   # None = auto-detect the only *.msc file present

# ── The known 10x -> real-imaging-objective calibration shift ─────────────
# Externally measured (comparing a handful of real-imaging-objective FOVs
# against the same tissue features in the 10x scan) -- NOT computed by this
# notebook, just applied. Added to every LOW-MAG tile's own (x_um, y_um).
LOW_MAG_OBJECTIVE  = "10x"
HIGH_MAG_OBJECTIVE = "60x"
SHIFT_DX_UM = 410.0
SHIFT_DY_UM = 420.0

# ── Segmentation parameters -- copied verbatim from this sample's own local
# 02_create_boundary_from_mosaic.ipynb (MERci/notebooks/prepare_imaging/
# lineage_tracing/merfish/), NOT re-derived, so the corrected boundary is
# produced the same way the original (uncorrected) one was. ──────────────
MOSAIC_KEEP_OBJECTIVES = [LOW_MAG_OBJECTIVE]   # exclude the high-mag calibration tiles from segmentation
WORKING_PIXEL_UM       = 5.0
THRESHOLD              = 400
SMOOTH_SIGMA_UM        = 10.0
CLOSE_RADIUS_UM        = 50.0
OPEN_RADIUS_UM         = 15.0
MARGIN_UM              = 25.0
MIN_TISSUE_AREA_UM2    = 4_000_000.0
MIN_HOLE_AREA_UM2      = 12_000.0
MIN_ISLAND_AREA_UM2    = 50_000.0
SIMPLIFY_TOL_UM        = 15.0

# ── FOV-grid parameters -- copied verbatim from this sample's own local
# 02_create_positions_from_boundaries.ipynb. ──────────────────────────────
MICROSCOPE           = "ST2"
non_overlap_fraction = 0.9
SCAN_DIRECTION       = "vertical"
pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE)
step_size_um = pixel_size_um * image_size_px * non_overlap_fraction
fov_size_um  = pixel_size_um * image_size_px

# ── Display normalization for the composited both-objectives views (steps
# 1-2 below) -- the two objectives are shot at very different exposures, so
# a single shared percentile-based stretch over the composited canvas lets
# the high-mag patch's saturated values wash out any real visual comparison.
# Each objective's own tiles are independently rescaled onto a common
# display range before compositing instead: low-mag keeps its own natural
# contrast range, high-mag is stretched from its own (narrower) range onto
# the SAME target scale. Display-only -- the real segmentation/grid steps
# below (4 onward) use the low-mag tiles at their own real, un-rescaled
# pixel values, exactly like the local 02_create_boundary_from_mosaic.ipynb.
LOW_MAG_DISPLAY_PCT  = (1.0, 99.0)
HIGH_MAG_DISPLAY_PCT = (2.0, 95.0)

CURRENT_POSITIONS_PATH = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
ADDED_POSITIONS_PATH   = POSITIONS_DIR / f"positions_{POSITIONS_TAG}_added.txt"

print(f"Sample name        : {SAMPLE_NAME}")
print(f"step_size_um       : {step_size_um:.1f}")
print(f"fov_size_um        : {fov_size_um:.1f}")
print(f"Shift (~{SHIFT_DX_UM/step_size_um:.2f} x {SHIFT_DY_UM/step_size_um:.2f} FOVs): "
      f"dX={SHIFT_DX_UM}, dY={SHIFT_DY_UM} um")
print(f"Current positions  : {CURRENT_POSITIONS_PATH}")
print(f"Will write         : {ADDED_POSITIONS_PATH}")

## 3 — Load the raw Steve mosaic (cached -- slow over a network drive)

Reading every tile's own `.stv` pickle is many small file opens over
(typically) a network-mounted acquisition drive -- slow regardless of total
data size (dominated by per-file latency, not throughput). Cached to
`analysis/cache/fix_mosaic_shift_missing_fovs/steve_tiles.pkl`, invalidated
by the `.msc` manifest's own mtime, per `NOTEBOOK_GUIDELINES.md` #2/#3.

In [ ]:
msc_candidates = sorted(MOSAIC_DIR.glob(f"{MOSAIC_NAME or '*'}.msc"))
if not msc_candidates:
    raise FileNotFoundError(f"No .msc mosaic manifest found in {MOSAIC_DIR}.")
if len(msc_candidates) > 1:
    print(f"WARNING: {len(msc_candidates)} .msc files found, using the first: {msc_candidates[0].name}.")
MSC_PATH = msc_candidates[0]

tiles_cache = CACHE_DIR / "steve_tiles.pkl"
msc_mtime   = MSC_PATH.stat().st_mtime

cached_ok = False
if tiles_cache.exists():
    with open(tiles_cache, "rb") as fh:
        cached = pickle.load(fh)
    cached_ok = cached.get("msc_mtime") == msc_mtime
    if cached_ok:
        tiles_all = cached["tiles"]
        print(f"Loaded {len(tiles_all)} cached tile(s): {tiles_cache}")

if not cached_ok:
    print(f"Reading {MSC_PATH} (every tile's own .stv file -- can take a couple of minutes "
          f"over a network drive, cached afterward)...")
    tiles_all = load_steve_mosaic(MSC_PATH)
    with open(tiles_cache, "wb") as fh:
        pickle.dump({"msc_mtime": msc_mtime, "tiles": tiles_all}, fh)
    print(f"Loaded and cached {len(tiles_all)} tile(s): {tiles_cache}")

obj_counts = Counter(t.objective_name for t in tiles_all)
print(f"Objective breakdown: {dict(obj_counts)}")
for obj in (LOW_MAG_OBJECTIVE, HIGH_MAG_OBJECTIVE):
    if obj not in obj_counts:
        raise ValueError(f"No tiles found for objective={obj!r} -- check LOW_MAG_OBJECTIVE/"
                          f"HIGH_MAG_OBJECTIVE against the breakdown above.")

## 4 — Step 1: raw mosaic, both objectives together

Composites every tile (both objectives, UNSHIFTED) into one image -- since
the high-mag calibration tiles' own `zvalue` is always higher than every
low-mag tile's (Steve's own acquisition-order stacking, confirmed directly
from the loaded tiles), the high-mag patch paints on top wherever they
overlap. Whether the two objectives actually agree on where the tissue is
should be visible right away, from real tissue content -- not just tile
CENTER coordinates, which are indistinguishable at this whole-mosaic scale
for a shift of only a few hundred um.

The two objectives were shot at very different exposures -- composited at
their own raw values, the high-mag patch is badly saturated relative to
the low-mag background, which would make the comparison unreadable.
`normalize_tile_intensities` below independently rescales each objective's
own tiles onto a shared display range first (`LOW_MAG_DISPLAY_PCT`/
`HIGH_MAG_DISPLAY_PCT`, section 2) -- a DISPLAY-only transform; segmentation
in step 4 below reads the real, un-rescaled low-mag pixel values.

In [ ]:
low_tiles  = [t for t in tiles_all if t.objective_name == LOW_MAG_OBJECTIVE]
high_tiles = [t for t in tiles_all if t.objective_name == HIGH_MAG_OBJECTIVE]


def normalize_tile_intensities(tiles, low_pct, high_pct, target_lo=0.0, target_hi=1000.0):
    # Rescale every tile's own raw pixel values from this GROUP's pooled
    # [low_pct, high_pct] percentile range onto [target_lo, target_hi] --
    # display-only (returns new tile copies; originals untouched), so tiles
    # shot at very different exposures composite onto one comparable scale.
    sample = np.concatenate([t.image[::4, ::4].ravel() for t in tiles])
    lo, hi = np.percentile(sample, [low_pct, high_pct])
    if hi <= lo:
        return list(tiles)
    rescaled = []
    for t in tiles:
        norm = np.clip((t.image.astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)
        rescaled.append(dataclasses.replace(t, image=norm * (target_hi - target_lo) + target_lo))
    return rescaled


low_tiles_norm  = normalize_tile_intensities(low_tiles,  *LOW_MAG_DISPLAY_PCT)
high_tiles_norm = normalize_tile_intensities(high_tiles, *HIGH_MAG_DISPLAY_PCT)

raw_both_canvas = assemble_mosaic_canvas(low_tiles_norm + high_tiles_norm, working_pixel_um=WORKING_PIXEL_UM)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(raw_both_canvas.image, cmap="gray", vmin=0, vmax=1000)
ax.set_title(f"{SAMPLE_NAME}: raw mosaic, both objectives (uncorrected)")
ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step1_raw_mosaic_both_objectives.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step1_raw_mosaic_both_objectives.png'}")

## 5 — Step 2: shifted mosaic, both objectives together

Same composite as step 1, but with the known `(SHIFT_DX_UM, SHIFT_DY_UM)`
correction applied to every LOW-MAG tile's position first (the high-mag
tiles are the fixed calibration reference and never move). If the shift is
right, the high-mag patch should now look like a seamless, continuous part
of the surrounding low-mag tissue instead of a visibly separate/offset
patch.

In [ ]:
low_tiles_shifted = [dataclasses.replace(t, x_um=t.x_um + SHIFT_DX_UM, y_um=t.y_um + SHIFT_DY_UM)
                     for t in low_tiles]
low_tiles_shifted_norm = normalize_tile_intensities(low_tiles_shifted, *LOW_MAG_DISPLAY_PCT)

shifted_both_canvas = assemble_mosaic_canvas(low_tiles_shifted_norm + high_tiles_norm, working_pixel_um=WORKING_PIXEL_UM)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(shifted_both_canvas.image, cmap="gray", vmin=0, vmax=1000)
ax.set_title(f"{SAMPLE_NAME}: shifted mosaic, both objectives (dX={SHIFT_DX_UM}, dY={SHIFT_DY_UM} um)")
ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step2_shifted_mosaic_both_objectives.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step2_shifted_mosaic_both_objectives.png'}")
print("Visually confirm the high-mag patch now looks like a seamless part of the "
      "surrounding low-mag tissue (compare against step1's unshifted version) before "
      "trusting anything below -- if not, SHIFT_DX_UM/SHIFT_DY_UM (or their sign) "
      "need correcting. If in doubt, flag it rather than guessing.")

## 6 — Step 3: shifted mosaic canvas + current (already-imaging) positions overlay

Assembles the mosaic from the SHIFTED low-mag tiles only, at their own REAL
(un-normalized) pixel values (same `MOSAIC_KEEP_OBJECTIVES`/
`WORKING_PIXEL_UM` convention as the local `02_create_boundary_from_
mosaic.ipynb` -- this canvas also feeds segmentation in step 4), then
overlays every FOV in the positions file currently being imaged as its own
real PERIMETER square (not just a center point) -- if the original
boundary really was derived from the unshifted mosaic, this should show
the current FOV grid sitting offset from the real tissue.

In [ ]:
shifted_canvas = assemble_mosaic_canvas(
    [t for t in low_tiles_shifted if t.objective_name in MOSAIC_KEEP_OBJECTIVES],
    working_pixel_um=WORKING_PIXEL_UM,
)

current_positions = np.loadtxt(CURRENT_POSITIONS_PATH, delimiter=",")
print(f"Current positions: {len(current_positions)} FOV(s) from {CURRENT_POSITIONS_PATH.name}")

def _um_to_px(canvas, x, y):
    return ((np.asarray(x) - canvas.origin_um[0]) / canvas.pixel_size_um,
             (np.asarray(y) - canvas.origin_um[1]) / canvas.pixel_size_um)

fig, ax = plt.subplots(figsize=(9, 9))
covered_vals = shifted_canvas.image[shifted_canvas.covered]
vmin, vmax = np.percentile(covered_vals, [1, 99]) if covered_vals.size else (0, 1)
ax.imshow(shifted_canvas.image, cmap="gray", vmin=vmin, vmax=vmax)

half_px = (fov_size_um / 2) / shifted_canvas.pixel_size_um
px, py = _um_to_px(shifted_canvas, current_positions[:, 0], current_positions[:, 1])
for x, y in zip(px, py):
    ax.add_patch(mpatches.Rectangle((x - half_px, y - half_px), 2 * half_px, 2 * half_px,
                                     linewidth=0.4, edgecolor="yellow", facecolor="none"))
ax.plot([], [], color="yellow", label=f"current positions ({len(current_positions)})")
ax.set_title(f"{SAMPLE_NAME}: shifted (corrected) mosaic vs. currently-imaging FOV grid")
ax.legend(); ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step3_shifted_mosaic_vs_current_fovs.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step3_shifted_mosaic_vs_current_fovs.png'}")

## 7 — Step 4: re-segment the shifted mosaic, find FOVs using the SAME grid as OLD

Same `segment_mosaic_tissue` parameters as the local `02_create_boundary_
from_mosaic.ipynb`. For the FOV grid, this does NOT build an independent
grid centered on the new (shifted) tissue polygon's own bounding box --
`create_grid_positions` centers a fresh grid on whatever polygon it's given,
and the shift (~2.25 x 2.31 FOV widths) isn't an exact integer number of
FOVs, so an independently-centered new grid would share almost no exact
coordinates with the current positions file even where they really overlap
(forcing an approximate, distance-based comparison). Instead: build ONE
dense grid centered on the OLD tissue boundary's own bounding box (the
EXACT phase the current positions file's own grid already uses), sized to
cover both the old and shifted/new tissue regions, then FILTER that same
dense grid twice -- once against the OLD boundary+holes (sanity check:
should reproduce `current_positions` count almost exactly) and once against
the NEW (shifted) boundary+holes. Any FOV the two filtered sets share is
then an EXACT coordinate match, not an approximate overlap -- holes are
loaded from the EXISTING `positions/boundaries/from_mosaic/hole*.txt` files
(holes don't move; only the outer tissue boundary was ever derived from the
un-shifted mosaic's own threshold trace) and shifted the same way.

In [ ]:
BOUNDARY_DIR = POSITIONS_DIR / "boundaries" / "from_mosaic"

old_boundary_polygon = load_boundary_polygon(BOUNDARY_DIR / "boundary_positions.txt")
holes_raw = load_hole_polygons(BOUNDARY_DIR)
holes_shifted = [translate(h, xoff=SHIFT_DX_UM, yoff=SHIFT_DY_UM) for h in holes_raw]
print(f"Loaded OLD boundary + {len(holes_raw)} hole polygon(s)")

segmentation = segment_mosaic_tissue(
    shifted_canvas,
    threshold           = THRESHOLD,
    smooth_sigma_um     = SMOOTH_SIGMA_UM,
    close_radius_um     = CLOSE_RADIUS_UM,
    open_radius_um      = OPEN_RADIUS_UM,
    margin_um           = MARGIN_UM,
    min_tissue_area_um2 = MIN_TISSUE_AREA_UM2,
    min_hole_area_um2   = MIN_HOLE_AREA_UM2,
    min_island_area_um2 = MIN_ISLAND_AREA_UM2,
    simplify_tol_um     = SIMPLIFY_TOL_UM,
)
print(f"Threshold used : {segmentation.threshold:.1f}")
print(f"Tissue pieces  : {len(segmentation.tissue_polygons)}")

if len(segmentation.tissue_polygons) != 1:
    raise ValueError(
        f"Expected exactly 1 tissue piece for this sample's known single-boundary "
        f"layout, got {len(segmentation.tissue_polygons)} -- inspect the plot below "
        f"and adjust THRESHOLD/morphology parameters before continuing."
    )
new_tissue_polygon = segmentation.tissue_polygons[0]

ax = plot_mosaic_segmentation(shifted_canvas, segmentation)
ax.figure.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step4_new_segmentation.png", dpi=150)
plt.show()

# ONE dense grid, phase-locked to the OLD boundary's own bounding-box center,
# sized to cover both the old and new (shifted) tissue regions.
xmin_old, ymin_old, xmax_old, ymax_old = old_boundary_polygon.bounds
cx_old, cy_old = (xmin_old + xmax_old) / 2.0, (ymin_old + ymax_old) / 2.0
xmin_new, ymin_new, xmax_new, ymax_new = new_tissue_polygon.bounds
half_w = max(xmax_old - cx_old, xmax_new - cx_old, cx_old - xmin_old, cx_old - xmin_new)
half_h = max(ymax_old - cy_old, ymax_new - cy_old, cy_old - ymin_old, cy_old - ymin_new)
grid_extent_polygon = box(cx_old - half_w, cy_old - half_h, cx_old + half_w, cy_old + half_h)

grid, _, _ = create_grid_positions(grid_extent_polygon, step_size_um, direction=SCAN_DIRECTION)
dense_path = generate_scanning_path(grid, direction=SCAN_DIRECTION)

old_grid_positions = filter_scanning_path(dense_path, old_boundary_polygon, holes_raw, fov_size_um)
new_grid_positions = filter_scanning_path(dense_path, new_tissue_polygon, holes_shifted, fov_size_um)

print(f"Old boundary, from the shared grid : {len(old_grid_positions)} FOV(s) "
      f"(current positions file has {len(current_positions)})")
print(f"New (shifted) boundary, from the shared grid: {len(new_grid_positions)} FOV(s)")

## 8 — Step 5: overlay the OLD vs. NEW tissue boundary outlines

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.plot(*old_boundary_polygon.exterior.xy, "-",  lw=1.2, c="tab:orange", label="OLD boundary")
ax.plot(*new_tissue_polygon.exterior.xy,   "--", lw=1.2, c="tab:green",  label="NEW (shifted) boundary")
for h in holes_raw:
    ax.plot(*h.exterior.xy, "-", lw=0.6, c="0.6")
for h in holes_shifted:
    ax.plot(*h.exterior.xy, "--", lw=0.6, c="0.4")
ax.set_title(f"{SAMPLE_NAME}: OLD vs. NEW tissue boundary")
ax.legend(); ax.axis("equal"); ax.invert_yaxis()
ax.set_xlabel("stage x (um)"); ax.set_ylabel("stage y (um)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step5_old_vs_new_boundary.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step5_old_vs_new_boundary.png'}")

## 9 — Step 6: classify NEW FOVs as already-covered vs. MISSING

Since `old_grid_positions` and `new_grid_positions` (section 7) come from
filtering the exact SAME dense grid, a FOV present in both is an EXACT
coordinate match -- no distance-based fuzzy matching needed. MISSING = FOVs
in the new (shifted) boundary's filtered set that are NOT in the old
boundary's filtered set: real tissue area the old (mis-positioned) grid
never imaged. Drawn as real FOV perimeter squares over both boundary
outlines, not center points.

In [ ]:
def _coord_set(coords, decimals=3):
    return {(round(float(x), decimals), round(float(y), decimals)) for x, y in coords}


old_set = _coord_set(old_grid_positions)
missing_mask = np.array([tuple(np.round(p, 3)) not in old_set for p in new_grid_positions])
missing_coords_unordered = new_grid_positions[missing_mask]

print(f"NEW (shifted boundary) FOVs           : {len(new_grid_positions)}")
print(f"Already covered (exact match with OLD): {(~missing_mask).sum()}")
print(f"MISSING (no exact match)              : {missing_mask.sum()}")

fig, ax = plt.subplots(figsize=(9, 9))
ax.plot(*old_boundary_polygon.exterior.xy, "-",  lw=1, c="0.5",      label="OLD boundary")
ax.plot(*new_tissue_polygon.exterior.xy,   "--", lw=1, c="tab:blue", label="NEW boundary")
for h in holes_raw:
    ax.plot(*h.exterior.xy, "-", lw=0.5, c="0.7")

half_um = fov_size_um / 2
for x, y in new_grid_positions[~missing_mask]:
    ax.add_patch(mpatches.Rectangle((x - half_um, y - half_um), fov_size_um, fov_size_um,
                                     linewidth=0.3, edgecolor="tab:green", facecolor="none"))
for x, y in missing_coords_unordered:
    ax.add_patch(mpatches.Rectangle((x - half_um, y - half_um), fov_size_um, fov_size_um,
                                     linewidth=0.4, edgecolor="tab:red", facecolor="tab:red", alpha=0.3))
ax.plot([], [], color="tab:green", label=f"already covered ({(~missing_mask).sum()})")
ax.plot([], [], color="tab:red",   label=f"MISSING ({missing_mask.sum()})")
ax.set_title(f"{SAMPLE_NAME}: missing FOVs (never imaged by the old grid)")
ax.legend(); ax.axis("equal"); ax.invert_yaxis()
ax.set_xlabel("stage x (um)"); ax.set_ylabel("stage y (um)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step6_missing_fovs.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step6_missing_fovs.png'}")

## 10 — Step 7: re-order the MISSING FOVs into their own short-travel loop

Simply keeping `new_grid_positions`'s own boustrophedon order (inherited
from `dense_path`, section 7) for JUST the missing subset does NOT give a
sensible loop: that order snakes through the WHOLE tissue column by
column, and any single column can contribute anywhere from zero to all of
its FOVs to the missing set depending on where that column's own boundary
happens to differ from the old grid -- extracting a boolean-masked
subsequence from that full snake jumps unpredictably between whichever
fragments happen to be missing in each column, in column order, not in a
locally-short path.

Tried a simple per-column boustrophedon re-sort first (group by real
lattice column, sort each column by the other axis, alternate direction
column to column -- the same convention `generate_scanning_path` uses for
a full grid). Measuring actual total travel (`get_path_stats`) showed this
was NOT reliably better than the naive subsequence order: a single lattice
column can itself contain more than one disconnected run of missing FOVs
(e.g. two separate notches crossing the same column), and sorting that
column's points by one axis alone still jumps across the gap between runs.
**Used a greedy nearest-neighbor walk instead** -- starting from the
missing FOV closest to `current_positions`'s own last point (so the
transit from the existing loop's end into this new loop is also short, not
just the loop's own internal travel), then repeatedly stepping to the
nearest not-yet-visited missing FOV. Simple and never takes a large hop
when a smaller one is available; not a guaranteed-optimal tour (true
optimal touring is NP-hard), but directly measured below to confirm it
beats both alternatives on this real data before using it.

In [ ]:
def nearest_neighbor_order(coords, start_point):
    remaining = coords.copy()
    order = []
    current = np.asarray(start_point, dtype=float)
    while len(remaining):
        dists = np.linalg.norm(remaining - current, axis=1)
        idx = int(np.argmin(dists))
        current = remaining[idx]
        order.append(current)
        remaining = np.delete(remaining, idx, axis=0)
    return np.array(order)


def boustrophedon_order(coords, step_size_um, direction="vertical"):
    primary_axis, secondary_axis = (0, 1) if direction == "vertical" else (1, 0)
    col_idx = np.round((coords[:, primary_axis] - coords[:, primary_axis].min()) / step_size_um).astype(int)
    chunks = []
    for col in sorted(set(col_idx)):
        members = coords[col_idx == col]
        members = members[np.argsort(members[:, secondary_axis])]
        if col % 2 == 1:
            members = members[::-1]
        chunks.append(members)
    return np.concatenate(chunks, axis=0)


missing_coords_column_sorted = boustrophedon_order(missing_coords_unordered, step_size_um, direction=SCAN_DIRECTION)
missing_coords_nearest_neighbor = nearest_neighbor_order(missing_coords_unordered, current_positions[-1])

candidates = {
    "new_grid_positions' own filtered order": missing_coords_unordered,
    "per-column boustrophedon re-sort":       missing_coords_column_sorted,
    "greedy nearest-neighbor walk":           missing_coords_nearest_neighbor,
}
for label, coords in candidates.items():
    length, max_step = get_path_stats(coords)
    print(f"{label:38s}: {length/1000:6.2f} mm total (max single step {max_step:6.0f} um)")

missing_coords = missing_coords_nearest_neighbor
print(f"\nUsing: greedy nearest-neighbor walk")

## 11 — Step 8-9: append the re-ordered MISSING FOVs, save a new positions file

Appended directly after the current (OLD) positions, so the existing
imaging loop's own order is completely undisturbed; only a new loop over
the appended tail needs to be added in Dave.

Writes a NEW file (`_added` suffix) -- never overwrites the original
positions file this sample is currently being imaged from.

In [ ]:
added_positions = np.concatenate([current_positions, missing_coords], axis=0)
save_positions_array(added_positions, ADDED_POSITIONS_PATH)

print(f"Wrote {len(added_positions)} FOV(s) ({len(current_positions)} original + "
      f"{len(missing_coords)} appended missing) to:")
print(f"  {ADDED_POSITIONS_PATH}")

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(current_positions[:, 0], current_positions[:, 1],
           s=6, c="tab:orange", label=f"original ({len(current_positions)})")
ax.scatter(missing_coords[:, 0], missing_coords[:, 1],
           s=10, c="tab:red", label=f"appended, missing ({len(missing_coords)})")
ax.plot(missing_coords[:, 0], missing_coords[:, 1], "-", lw=0.5, c="tab:red", alpha=0.5)
ax.set_title(f"{SAMPLE_NAME}: final positions file ({ADDED_POSITIONS_PATH.name})")
ax.legend(); ax.axis("equal"); ax.invert_yaxis()
ax.set_xlabel("stage x (um)"); ax.set_ylabel("stage y (um)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step7_final_added_positions.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step7_final_added_positions.png'}")
print()
print("Review every figure in", FIGURES_DIR, "before using this file -- in particular "
      "step2 (does the shift actually align the two objectives?) and step6/7 (do the "
      "MISSING FOVs look like a real tissue-edge strip, not noise).")